In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import math

In [2]:
weather_train_df = pd.read_csv("Datast/weather_train.csv")

In [3]:
wet_cop=weather_train_df.copy()

In [ ]:
wet_cop.hist(bins=100,figsize=(20,20))

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# col=wet_cop.select_dtypes(include=['float64', 'int64']).columns
# # Select only numerical columns
# # numeric_cols = cp_main.select_dtypes(include=["number"]).columns
# # col=['meter_reading', 'square_feet', 'year_built', 'floor_count','air_temperature', 'cloud_coverage', 'dew_temperature','precip_depth_1_hr', 'sea_level_pressure', 'wind_direction','wind_speed']

# # Plot each column separately
# for col in col:
#     plt.figure(figsize=(6, 4))
#     sns.boxplot(x=wet_cop[col])
#     plt.title(f"Boxplot of {col}")
#     plt.show()
# sns.boxplot(x=wet_cop['relative_humidity'])

In [4]:
def saturation_vapor_pressure(temperature):
    return 6.112 * math.exp((17.67 * temperature) / (temperature + 243.5))

# Function to calculate relative humidity
def relative_humidity(air_temp, dew_temp):
    E_air = saturation_vapor_pressure(air_temp)
    E_dew = saturation_vapor_pressure(dew_temp)
    RH = (E_dew / E_air) * 100
    return RH

# Apply the function to each row in the dataframe to calculate relative humidity
wet_cop['relative_humidity'] = wet_cop.apply(lambda row: relative_humidity(row['air_temperature'], row['dew_temperature']), axis=1)
# test_wet_cop['relative_humidity'] = test_wet_cop.apply(lambda row: relative_humidity(row['air_temperature'], row['dew_temperature']), axis=1)

In [5]:
# Simplified heat index (Rothfusz regression approximation)
wet_cop['heat_index'] = (
    0.5 * (wet_cop['air_temperature'] + 61.0 + 
    (wet_cop['air_temperature'] - 68.0) * 1.2 + 
    wet_cop['dew_temperature'] * 0.094
)
)

# Wind chill (for temps < 10°C and wind > 4.8 km/h)
mask = (wet_cop['air_temperature'] < 10) & (wet_cop['wind_speed'] > 1.34)  # 1.34 m/s ≈ 4.8 km/h
wet_cop['wind_chill'] = (
    13.12 + 0.6215 * wet_cop['air_temperature'] - 
    11.37 * (wet_cop['wind_speed'] ** 0.16) + 
    0.3965 * wet_cop['air_temperature'] * (wet_cop['wind_speed'] ** 0.16)
)
wet_cop['wind_chill'] = wet_cop['wind_chill'].where(mask, wet_cop['air_temperature'])

wet_cop['feels_like'] = np.where(
    wet_cop['air_temperature'] >= 27,  # Hot threshold
    wet_cop['heat_index'],
    np.where(
        wet_cop['air_temperature'] <= 10,  # Cold threshold
        wet_cop['wind_chill'],
        wet_cop['air_temperature']  # Default
    )
)

In [6]:
import numpy as np
import pandas as pd

# Assuming wet_cop is your weather dataframe
wet_cop["timestamp"] = pd.to_datetime(wet_cop["timestamp"])

# Extract basic time features
wet_cop["hour"] = wet_cop["timestamp"].dt.hour
wet_cop["day_of_week"] = wet_cop["timestamp"].dt.weekday  # Monday=0, Sunday=6
wet_cop["month"] = wet_cop["timestamp"].dt.month
wet_cop["day_of_year"] = wet_cop["timestamp"].dt.dayofyear
wet_cop["is_weekend"] = (wet_cop["day_of_week"] >= 5).astype(int)

# Cyclical encoding for hour
wet_cop["hour_sin"] = np.sin(2 * np.pi * wet_cop["hour"] / 24)
wet_cop["hour_cos"] = np.cos(2 * np.pi * wet_cop["hour"] / 24)

# Cyclical encoding for day of the week
wet_cop["day_sin"] = np.sin(2 * np.pi * wet_cop["day_of_week"] / 7)
wet_cop["day_cos"] = np.cos(2 * np.pi * wet_cop["day_of_week"] / 7)

# Cyclical encoding for month
wet_cop["month_sin"] = np.sin(2 * np.pi * wet_cop["month"] / 12)
wet_cop["month_cos"] = np.cos(2 * np.pi * wet_cop["month"] / 12)

# Season encoding (0: Winter, 1: Spring, 2: Summer, 3: Fall)
wet_cop["season"] = wet_cop["month"].map(lambda m: 0 if m in [12, 1, 2] else
                                         1 if m in [3, 4, 5] else
                                         2 if m in [6, 7, 8] else 3)

# Drop original timestamp column if no longer needed
wet_cop.drop(columns=["timestamp"], inplace=True)

# Display updated dataframe
wet_cop.head()


,site_id,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed,relative_humidity,heat_index,...,month,day_of_year,is_weekend,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos,season
0,0,25.0,6.0,20.0,NaN,1019.7,0.0,0.0,73.780558,18.1400,...,1,1,0,0.000000,1.000000,-0.433884,-0.900969,0.5,0.866025,0
1,0,24.4,NaN,21.1,-1.0,1020.2,70.0,1.5,81.848292,17.5317,...,1,1,0,0.258819,0.965926,-0.433884,-0.900969,0.5,0.866025,0
2,0,22.8,2.0,21.1,0.0,1020.2,0.0,0.0,90.139994,15.7717,...,1,1,0,0.500000,0.866025,-0.433884,-0.900969,0.5,0.866025,0
3,0,21.1,2.0,20.6,0.0,1020.1,0.0,0.0,96.968347,13.8782,...,1,1,0,0.707107,0.707107,-0.433884,-0.900969,0.5,0.866025,0
4,0,20.0,2.0,20.0,-1.0,1020.0,250.0,2.6,100.000000,12.6400,...,1,1,0,0.866025,0.500000,-0.433884,-0.900969,0.5,0.866025,0


In [ ]:
wet_cop.columns

In [8]:
# monthly_readings = wet_cop.groupby("month")["sea_level_pressure"].mean()

# # Plot
# plt.figure(figsize=(10, 5))
# sns.lineplot(x=monthly_readings.index, y=monthly_readings.values, marker="o", linestyle="-", color="b")

# # Labels and title
# plt.xlabel("Month", fontsize=12)
# plt.ylabel("Average feels_like", fontsize=12)
# plt.title("Monthly feels_like Trends", fontsize=14)
# plt.xticks(range(1, 13), ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
# plt.grid(True)

# # Show plot
# plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numeric_cols = wet_cop.select_dtypes(include=['float64', 'int64']).columns
corr_df = wet_cop[numeric_cols].corr()

plt.figure(figsize=(20, 16))
sns.heatmap(
    corr_df,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    mask=np.triu(np.ones_like(corr_df))  # Hide upper triangle for clarity
)
plt.title("Feature Correlation Heatmap")
plt.show()

In [9]:
T_min = -10  # Below this, heating load is maxed
T_max = 35   # Above this, cooling load is maxed

wet_cop['feels_like_capped'] = wet_cop['feels_like'].clip(T_min, T_max)
wet_cop['feels_like_capped']

0         25.000000
1         24.400000
2         22.800000
3         21.100000
4         20.000000
            ...    
139768     1.534909
139769     0.637652
139770     1.544926
139771     1.086355
139772    -0.970798
Name: feels_like_capped, Length: 139773, dtype: float64

In [10]:

wet_cop['precip_depth_1_hr'] = (
    wet_cop['precip_depth_1_hr']
    .replace(-1.0, np.nan)
    .interpolate(method='linear', limit=6)  # 6-hour gap limit
    .fillna(0)  # Fill remaining gaps with 0
)

In [11]:
bins = [-0.1, 0.1, 5.0, 15.0, float('inf')]  # -0.1 to catch near-zero values  
labels = ['No Rain', 'Light Rain', 'Moderate Rain', 'Heavy Rain']  

wet_cop['precip_1h_category'] = pd.cut(
    wet_cop['precip_depth_1_hr'],  
    bins=bins,  
    labels=labels  
)

In [12]:
wet_cop['is_light_rain'] = (wet_cop['precip_depth_1_hr'] > 0.1).astype(int)  

wet_cop['is_moderate_rain'] = ((wet_cop['precip_depth_1_hr'] > 5.0) & (wet_cop['precip_depth_1_hr'] < 15.0)).astype(int)

wet_cop['is_heavy_rain'] = (wet_cop['precip_depth_1_hr'] >= 15.0).astype(int)  

In [ ]:
curr_imp_feat=['feels_like_capped','relative_humidity','site_id',"season",'sea_level_pressure','wind_direction','hour', 'day_of_week', 'month', 'day_of_year','is_weekend', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin','month_cos', 'season','is_light_rain', 'is_moderate_rain', 'is_heavy_rain']

# ['site_id', 'air_temperature', 'cloud_coverage', 'dew_temperature',
#        'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction',
#        'wind_speed', 'relative_humidity', 'heat_index', 'wind_chill',
#        'feels_like', 'hour', 'day_of_week', 'month', 'day_of_year',
#        'is_weekend', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin',
#        'month_cos', 'season', 'feels_like_capped', 'precip_1h_category',
#        'is_light_rain', 'is_moderate_rain', 'is_heavy_rain']

In [ ]:
# wet_cop['feels_like_capped']

In [ ]:
for col in curr_imp_feat:
    plt.figure(figsize=(4, 3))
    sns.boxplot(x=wet_cop[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

In [13]:
from scipy.stats.mstats import winsorize
# mean_slp = wet_cop["sea_level_pressure"].mean()
# std_slp = wet_cop["sea_level_pressure"].std()

# wet_cop["slp_zscore"] = (wet_cop["sea_level_pressure"] - mean_slp) / std_slp
# outliers = wet_cop[wet_cop["slp_zscore"].abs() > 3]  # Outside 3 standard deviations

# print("Number of Outliers:", len(outliers))
# outliers
# mean_slp
# len(wet_cop['sea_level_pressure'])

wet_cop["sea_level_pressure"] = winsorize(wet_cop["sea_level_pressure"], limits=[0.01, 0.01])  # Capping at 0.5% on both ends

In [ ]:
wet_cop.isna().sum()

In [ ]:
train_df=pd.read_csv('Datast/train.csv')
build_df=pd.read_csv('Datast/building_metadata.csv')

In [ ]:
# build_df.columns
merg=wet_cop.merge(build_df,how='left')

In [ ]:
merg.isna().sum()